# 14 · SkewerJAX and fake_spectra: how each stage scales

Notebook 13 times the two codes on one target. This notebook moves one parameter at a time around it and follows every
stage of the four entries (fake_spectra; SkewerJAX default and matched on the CPU; SkewerJAX default on the GPU), with the
calls and the protocol of notebook 13:

- **A, the simulation resolution:** TNG50-4, TNG50-3 and TNG50-2 (snapshot 25, $z = 3$) at $N_{\max}^2$ skewers;
- **B, the number of skewers:** TNG50-2, $n = 4, 8, \ldots, N_{\max}$ skewers per side;
- **C1, the pixel width:** TNG50-2, $N_{\max}^2$ skewers, pixels of 1, 2, 4, 8 and 16 km/s (SkewerJAX keeps its sampling
  of 0.25 km/s along the skewer);
- **C2, the sampling along the skewer** (SkewerJAX only): TNG50-2, $N_{\max}^2$ skewers, 1 km/s pixels, points every
  0.125 to 4 km/s; fake_spectra has no such parameter, so its 1 km/s run is the fixed reference.

$N_{\max} = 128$, the grid of notebook 13: half the side of the largest grid on which fake_spectra runs within 20 minutes
on this machine (256; SkewerJAX's dense $\tau$ sum on the CPU would take about 6 h per run there). The skewers run
along $y$ on the nested grid $x, z = iL/n$, so the **reference skewer** $(L/4, L/4)$, notebook 12's, belongs to every grid:
its flux, stored for every entry and point, is the precision check of every block.

**By default the notebook reads the records** of `scripts/benchmarks/benchmark_scaling.py` (`benchmarks/results/scaling/`,
one JSON per point: stage times, JIT time, page-cache state, the machine, the agreement over all skewers and the flux of the
reference skewer) and runs in about a minute. **`RUN = True` re-runs the benchmark** for the points without a record
(resumable; each point in its own process): about 22 h on this machine for all of them (the TNG50-2 points at $128^2$ skewers dominate).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")             # this kernel reads the records and plots; the benchmark workers set their own
import inspect, json, logging
logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (PALETTE_CYCLE, SCALING_ENTRY_STYLES, add_panel_letters, plot_flux_checks,
                                      plot_scaling_block, plot_stage_grid, use_backend, use_style)
from skewerjax.utils.paths import BENCHMARK_RECORDS_DIR, DATA_DIR
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # paper style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes

## The calls

Every point runs the four entries with exactly these calls (`skewerjax.comparison.scaling`), the calls of notebook 13:
fake_spectra on every skewer at once, timed from outside; SkewerJAX on batches of skewers (1024 at 0.25 km/s; 512 at 0.125 km/s), every
stage of every entry timed and summed over the batches. The warm-up is the first batch, run once before the timed run;
the JIT time of a $\tau$ sum is the warm-up minus the same batch timed. One timed run per point.

In [ ]:
from skewerjax.comparison import scaling as S
print(inspect.getsource(S.run_fake_spectra))
print(inspect.getsource(S.skewerjax_batch))

## The records

`RUN = False` (the default) reads the records; `True` measures every point without one, then reads them all (the same as
`python scripts/benchmarks/benchmark_scaling.py`). `N_MAX` is that of notebook 13.

In [ ]:
RUN = False                                    # True: run the benchmark for the points without a record (resumes)
N_MAX = S.N_MAX                                # skewers per side of the largest grid (notebook 13)
OUT = BENCHMARK_RECORDS_DIR / "scaling"        # one JSON per point ($SKEWERJAX_BENCHMARK_RESULTS)
POINTS = S.scaling_points(N_MAX)
if RUN:
    S.run_points(POINTS, OUT, data_dir=DATA_DIR)
R = S.load_records(OUT)
print(f"{len(R)} of {len(POINTS)} points recorded in {OUT}")
for c in POINTS:
    r = R.get(c["id"])
    if r is None:
        print(f"  {c['id']:28s} ({', '.join(c['blocks'])}): not recorded")
        continue
    print(f"  {c['id']:28s} ({', '.join(c['blocks']):9s}) {r['date']}, page cache: "
          + ", ".join(f"{k} {v:.0%} ({S.cache_label(v)})" for k, v in r['cache'].items()) + f"; {r['batches']['n']} batches")
m = next(iter(R.values()))["machine"] if R else {}
print(f"machine: {m.get('cpu')}, {m.get('logical_cpus')} logical CPUs, {m.get('ram_GB', float('nan')):.0f} GB RAM; GPU {next(iter(R.values()))['gpu'] if R else '-'}")

In [ ]:
ENTRIES = list(S.ENTRIES)
LABEL = {"A": "the simulation", "B": "the number of skewers", "C1": "the pixel width", "C2": "the sampling along the skewer"}
XLABEL = {"A": "gas cells in the snapshot", "B": "skewers", "C1": "pixel width [km/s]", "C2": "point spacing [km/s]"}


def block(b):
    """The records of block b in the order of its parameter, the parameter, and one row of stages per record."""
    recs = S.block_records(R, b, N_MAX)
    x = [S.block_parameter(b, r) for r in recs]
    rows = [{e: r["entries"][e]["stages"] for e in ENTRIES if e in r["entries"]} for r in recs]
    return recs, x, rows


def title(b, r):
    c = r["config"]
    return {"A": f"{c['sim']}: {int(r['snapshot']['n_gas']):,} cells", "B": f"{c['n']} x {c['n']} = {c['n_skewers']:,} skewers",
            "C1": f"pixels of {c['dv_kms']:g} km/s", "C2": f"points every {c['spacing_kms']:g} km/s"}[b]


def stage_figure(b):
    recs, x, rows = block(b)
    jit = [{e: r["entries"][e].get("jit_s", 0.0) for e in r["entries"]} for r in recs]
    fig, axs = plot_stage_grid(rows, labels=[title(b, r) for r in recs], jit=jit)
    add_panel_letters(axs, labels=[f"{chr(97 + i)}{j + 1}" for i in range(len(recs)) for j in range(3)])
    return fig, axs


def print_stats(stats):
    """The rms and max |dF| of every curve of a precision figure, over every pixel."""
    for (col, lab, line), (r, m) in stats.items():
        print(f"{col:42s} {lab:26s} {line:6s} rms {r:.1e}  max {m:.1e}")


def agreement_table(b):
    recs, x, _ = block(b)
    for r in recs:
        a = r.get("agreement", {})
        print(f"{title(b, r):34s} " + "; ".join(f"{k}: rms {v['rms_dF']:.1e} max {v['max_dF']:.1e}" for k, v in a.items()))


def have(b):
    """Whether block b has records to show (a partial RUN = True leaves some blocks empty: they are skipped)."""
    ok = len(S.block_records(R, b, N_MAX)) > 0
    if b == "C2":   # its fixed reference: fake_spectra at 1 km/s
        ok = ok and S.point_config("TNG50-2", N_MAX)["id"] in R
    if not ok:
        print(f"block {b}: no record yet (RUN = True measures it)")
    return ok

## A · The simulation resolution

TNG50-4, -3 and -2 hold 19.3 million, 152 million and 1.2 billion gas cells in boxes of the same size: the same skewers
cross 8 times more, smaller cells from one to the next.

**Where the time goes, point by point** (as notebook 13, one row per point: (a1) the stages, (a2) their shares, (a3) the totals and the speed-ups over fake_spectra; then (b1), ...).

In [ ]:
if have("A"):
    fig, axs = stage_figure("A")
    fig.suptitle("From TNG50-4 to TNG50-2 fake_spectra's run grows 18×, SkewerJAX's 1.3× (CPU) and 1.7× (GPU)", fontsize=21)
    plt.show()

In [ ]:
if have("A"):
    recs, x, rows = block("A")
    fig, axs = plot_scaling_block(x, rows, xlabel=XLABEL["A"])
    add_panel_letters(axs)
    fig.suptitle("fake_spectra scales with the cells (slope 0.69); SkewerJAX's τ sum does not see them (0.03)", fontsize=21)
    plt.show()
    agreement_table("A")

**Precision.** The flux of the reference skewer from every entry (top), and $|\Delta F|$ against fake_spectra for
$\mathrm{Ly}\alpha$ (middle) and Si III (bottom), one column per snapshot; the rms and the max over its pixels are printed
below the figure. This skewer has almost no Si III ($|\Delta F| < 10^{-6}$): the agreement over every skewer, printed
above, covers Si III. The GPU's line lies on the CPU's.

In [ ]:
if have("A"):
    recs, _, _ = block("A")
    cols = []
    for r in recs:
        fl, v = r["reference"]["flux"], (np.arange(r["pixels"]["nbins"]) + 0.5) * r["pixels"]["dv_kms"]
        ref = fl["fake_spectra"]
        cols.append(dict(title=title("A", r), v=v, style=SCALING_ENTRY_STYLES,
                         F={e: fl[e]["HI-LyA"] for e in ENTRIES if e in fl},
                         dF={e: (np.subtract(fl[e]["HI-LyA"], ref["HI-LyA"]), np.subtract(fl[e]["SiIII-1206"], ref["SiIII-1206"]))
                             for e in ENTRIES if e in fl and e != "fake_spectra"}))
    fig, axs, stats = plot_flux_checks(cols)
    add_panel_letters(axs)
    fig.suptitle("Matched = fake_spectra to rms ΔF ≤ 3e-7 on every snapshot; the default differs by its physics", fontsize=21)
    plt.show()
    print_stats(stats)

## B · The number of skewers

TNG50-2, from $4^2$ to $N_{\max}^2$ skewers.

**Where the time goes, point by point** (as notebook 13, one row per point: (a1) the stages, (a2) their shares, (a3) the totals and the speed-ups over fake_spectra; then (b1), ...).

In [ ]:
if have("B"):
    fig, axs = stage_figure("B")
    fig.suptitle("fake_spectra's run stays near 210 s up to 1,024 skewers; SkewerJAX's grows with them", fontsize=21)
    plt.show()

In [ ]:
if have("B"):
    recs, x, rows = block("B")
    fig, axs = plot_scaling_block(x, rows, xlabel=XLABEL["B"])
    add_panel_letters(axs)
    fig.suptitle("SkewerJAX ∝ N^0.9, fake_spectra nearly flat: SkewerJAX is faster below ~1,000 (CPU) and ~2,700 (GPU) skewers", fontsize=21)
    plt.show()
    agreement_table("B")

**Precision.** Each entry's reference skewer at every $n$ against its own result at $n = N_{\max}$: a skewer must
not depend on how many others are computed with it (top: the flux, below: $|\Delta F|$).

In [ ]:
if have("B"):
    recs, _, _ = block("B")
    last = recs[-1]
    v = (np.arange(last["pixels"]["nbins"]) + 0.5) * last["pixels"]["dv_kms"]
    cols = []
    for e in ENTRIES:
        ref = last["reference"]["flux"][e]
        styles = {}
        F, dF = {}, {}
        for k, r in enumerate(recs):
            fl = r["reference"]["flux"].get(e)
            if fl is None:
                continue
            lab = f"n = {r['config']['n']}"
            styles[lab] = dict(color=PALETTE_CYCLE[k % len(PALETTE_CYCLE)], lw=2.0)
            F[lab] = fl["HI-LyA"]
            if r is not last:
                dF[lab] = (np.subtract(fl["HI-LyA"], ref["HI-LyA"]), np.subtract(fl["SiIII-1206"], ref["SiIII-1206"]))
        cols.append(dict(title=f"{e}: against n = {last['config']['n']}", v=v, F=F, dF=dF, style=styles))
    fig, axs, stats = plot_flux_checks(cols)
    add_panel_letters(axs)
    fig.suptitle("The reference skewer does not depend on the grid around it", fontsize=21)
    plt.show()
    print_stats(stats)

## C1 · The pixel width

TNG50-2, $N_{\max}^2$ skewers, pixels of 1 to 16 km/s. SkewerJAX keeps its sampling of 0.25 km/s: only the $\tau$ sum sees
the pixels. The precision check compares every width with the 1 km/s result **rebinned to that width: the average of $F$
over each coarse pixel** (each 1 km/s pixel weighted by its overlap with it, `scaling.rebin_average`).

**Where the time goes, point by point** (as notebook 13, one row per point: (a1) the stages, (a2) their shares, (a3) the totals and the speed-ups over fake_spectra; then (b1), ...).

In [ ]:
if have("C1"):
    fig, axs = stage_figure("C1")
    fig.suptitle("Coarser pixels shrink SkewerJAX's τ sum; fake_spectra and the host stages stay", fontsize=21)
    plt.show()

In [ ]:
if have("C1"):
    recs, x, rows = block("C1")
    fig, axs = plot_scaling_block(x, rows, xlabel=XLABEL["C1"])
    add_panel_letters(axs)
    fig.suptitle("SkewerJAX's CPU τ sum ∝ Δv^-0.9; at 16 km/s the default is 0.53× (CPU) and 0.60× (GPU) fake_spectra's speed", fontsize=21)
    plt.show()
    agreement_table("C1")

**Precision.** The flux of the reference skewer at each pixel width (top), and $|\Delta F|$ of every entry against its
own 1 km/s flux averaged over the coarse pixels: the difference between computing on coarse pixels and averaging fine ones.

In [ ]:
if have("C1"):
    recs, _, _ = block("C1")
    fine = recs[0]                                          # 1 km/s
    nb1, dv1 = fine["pixels"]["nbins"], fine["pixels"]["dv_kms"]
    e1 = np.arange(nb1 + 1) * dv1                           # the edges of the 1 km/s pixels [km/s]
    cols = []
    for r in recs[1:]:
        nb, dvp = r["pixels"]["nbins"], r["pixels"]["dv_kms"]
        ec = np.arange(nb + 1) * dvp
        v = 0.5 * (ec[1:] + ec[:-1])
        fl = r["reference"]["flux"]
        reb = {e: {ln: S.rebin_average(fine["reference"]["flux"][e][ln], e1, ec) for ln in S.LINE_IDS} for e in ENTRIES if e in fl}
        cols.append(dict(title=title("C1", r), v=v, style=SCALING_ENTRY_STYLES,
                         F={e: fl[e]["HI-LyA"] for e in ENTRIES if e in fl},
                         dF={e: (np.subtract(fl[e]["HI-LyA"], reb[e]["HI-LyA"]), np.subtract(fl[e]["SiIII-1206"], reb[e]["SiIII-1206"]))
                             for e in ENTRIES if e in fl}))
    fig, axs, stats = plot_flux_checks(cols)
    add_panel_letters(axs)
    fig.suptitle("Against the 1 km/s flux averaged over the coarse pixels, every entry alike", fontsize=21)
    plt.show()
    print_stats(stats)

## C2 · The sampling along the skewer

TNG50-2, $N_{\max}^2$ skewers, 1 km/s pixels, points every 0.125 to 4 km/s: the pieces are cut at the ends of the dual
segments of the points, so their number, and the cost of the $\tau$ sum, follow the sampling; the resolution check of
`compute_optical_depth` (pieces wider than a quarter of the line width) is printed for every spacing. fake_spectra has no
such parameter: its 1 km/s run of block A is the fixed reference of the speed-ups (dotted line).

**Where the time goes, point by point** (as notebook 13, one row per point: (a1) the stages, (a2) their shares, (a3) the totals and the speed-ups over fake_spectra; then (b1), ...).

In [ ]:
if have("C2"):
    fig, axs = stage_figure("C2")
    fig.suptitle("The sampling sets SkewerJAX's τ sum on the CPU: 3,856 s at 0.125 km/s, 457 s at 4 km/s", fontsize=21)
    plt.show()

In [ ]:
if have("C2"):
    recs, x, rows = block("C2")
    fs_ref = R[S.point_config("TNG50-2", N_MAX)["id"]]["entries"]["fake_spectra"]["total"]
    fig, axs = plot_scaling_block(x, rows, xlabel=XLABEL["C2"], reference="fake_spectra", reference_total=fs_ref, entries=list(S.SKEWERJAX_ENTRIES))
    add_panel_letters(axs)
    fig.suptitle("SkewerJAX's τ sum ∝ spacing^-0.65 down to the cells; the host stages are a floor of ~800 s", fontsize=21)
    plt.show()
    for r in recs:
        res = r["resolution"]
        print(f"{title('C2', r):26s} resolution check: max W/b " + ", ".join(f"{m:.2f}" for m in res["max_ratio"])
              + "; column fraction above 0.25 b: " + ", ".join(f"{f:.1e}" for f in res["column_fraction"]) + ("; WARNED" if res["warned"] else "; passes"))

**Precision.** SkewerJAX's reference skewer at every spacing against its result at 0.125 km/s (top: the flux, with
fake_spectra's at 1 km/s in grey).

In [ ]:
if have("C2"):
    recs, _, _ = block("C2")
    finest = recs[0]                                        # 0.125 km/s
    v = (np.arange(finest["pixels"]["nbins"]) + 0.5) * finest["pixels"]["dv_kms"]
    fs_flux = R[S.point_config("TNG50-2", N_MAX)["id"]]["reference"]["flux"]["fake_spectra"]
    cols = []
    for e in S.SKEWERJAX_ENTRIES:
        styles, F, dF = {"fake_spectra (1 km/s)": dict(SCALING_ENTRY_STYLES["fake_spectra"])}, {"fake_spectra (1 km/s)": fs_flux["HI-LyA"]}, {}
        ref = finest["reference"]["flux"].get(e)
        if ref is None:
            continue
        for k, r in enumerate(recs):
            fl = r["reference"]["flux"].get(e)
            if fl is None:
                continue
            lab = f"{r['config']['spacing_kms']:g} km/s"
            styles[lab] = dict(color=PALETTE_CYCLE[k % len(PALETTE_CYCLE)], lw=2.0)
            F[lab] = fl["HI-LyA"]
            if r is not finest:
                dF[lab] = (np.subtract(fl["HI-LyA"], ref["HI-LyA"]), np.subtract(fl["SiIII-1206"], ref["SiIII-1206"]))
        cols.append(dict(title=f"{e}: against 0.125 km/s", v=v, F=F, dF=dF, style=styles))
    fig, axs, stats = plot_flux_checks(cols)
    add_panel_letters(axs)
    fig.suptitle("Against 0.125 km/s: rms ΔF 3e-6 at the default 0.25 km/s, 4e-4 at 4 km/s", fontsize=21)
    plt.show()
    print_stats(stats)

## Discussion

**What sets each code's time.** fake_spectra passes over every particle of the snapshot: reading and the gas physics
grow with the number of cells (block A: slopes 1.0 and 0.9; 30 s on TNG50-4, 526 s on TNG50-2) and hardly with the
skewers (block B: about 210 s from 16 to 1,024 skewers; only its search and $\tau$ sum grow beyond, slope 0.35). SkewerJAX
reads and processes the cells near the skewers only, so every stage grows with the skewers (block B: total slope 0.9 on
the CPU, 0.8 on the GPU) and its $\tau$ sum, the CPU's largest stage, does not see the cells at all (block A: slope 0.03):
its cost is the number of pieces, set by the sampling along the skewer (block C2) and the pixels (block C1).

**Where each code wins.** With few skewers SkewerJAX is far faster: $36\times$ (CPU) and $46\times$ (GPU) at 16 skewers of TNG50-2,
$4\times$ and $9\times$ at 256. fake_spectra wins beyond about 1,000 skewers on the CPU and 2,700 on the GPU; at $128^2$ it is $5.7\times$
faster than the CPU default and $2.4\times$ faster than the GPU. On smaller snapshots the crossing moves to fewer skewers
(at $128^2$ fake_spectra is $77\times$ faster than SkewerJAX's CPU on TNG50-4, $31\times$ on TNG50-3).

**The levers.** The GPU makes the $\tau$ sum about $7\times$ faster ($4\times$ at 16 km/s pixels), after which the host stages (selection, pieces,
I/O: about 960 s at $128^2$ on TNG50-2) set the time. Coarser pixels cut the CPU's $\tau$ sum as $\Delta v^{-0.9}$ (2,057 s at 1 km/s,
169 s at 16 km/s), at the price of the flux of coarse pixels against averaged fine ones (rms $\Delta F$ from 3e-4 at 2 km/s
to 1.4e-2 at 16 km/s, the same for fake_spectra). A sparser sampling cuts it as $(\text{spacing})^{-0.65}$ down to the size of the
cells (no gain beyond 2 km/s), at rms $\Delta F$ 1.3e-5 (0.5 km/s) to 4.4e-4 (4 km/s) against 0.125 km/s, and with the
resolution check of `compute_optical_depth` warning from 0.5 km/s on (Si III's pieces wider than a quarter of its line
width); the default 0.25 km/s passes it and is within rms 3e-6 of 0.125 km/s.

**The matched mode** reproduces fake_spectra over every skewer to rms $\Delta F \le 3\times10^{-7}$ on all three
snapshots and at every number of skewers ($\le 7\times10^{-7}$ at every pixel width); its top-hat chords make it the slowest entry on TNG50-2
(pieces 1,700–2,300 s at $128^2$, 3–$4\times$ the exact pieces). It does not depend on the sampling (its flux is the same to
4e-8 at every spacing); its stage times there spread from 1,281 to 3,016 s for the same work, the run-to-run noise of
this memory-heavy stage in a single timed run.

**Caveats.** One timed run per point (notebook 13 has three: its totals agree with this notebook's to 3%, except
fake_spectra's: 393 s there and 526 s here, its reads from a colder page cache). The snapshot of TNG50-2 is larger than the
RAM, so every point reads from the disk: the page cache held 0–32% of it before each read. The machine rebooted during
block C1: the points after it (16 km/s, and block C2) started from an empty page cache; their reads (SkewerJAX 78–116 s,
fake_spectra 169 s) fall within the spread of the others (SkewerJAX 85–122 s, fake_spectra 117–191 s).